# Active-set SQP and parametric continuation

This notebook is the end-to-end demo of pounce's Phase 5b/5c
warm-start machinery. We'll:

1. Solve a parametric NLP with the default IPM and capture the converged multipliers.
2. Classify the active set at that solution via `pounce.classify_working_set`.
3. Perturb the parameter, install the working set + previous primal as warm-start, and run the SQP corrector.
4. Sweep the parameter through 20 steps to compare cold-IPM, cold-SQP, and warm-SQP iteration counts.

The companion tutorial is at `docs/tutorials/active-set-sqp.md`.

In [1]:
import numpy as np
import pounce

print('pounce', pounce.__version__)

pounce 0.3.0


## The parametric NLP

Simplex projection — the canonical "active set rotates with the
parameter" benchmark:

$$x^\star(p) = \arg\min \;\tfrac{1}{2}\|x - p\|^2 \quad \text{s.t.} \quad \sum_i x_i = 1, \quad x \ge 0.$$

As $p$ rotates around the simplex centre the support of the
optimum (which $x_i = 0$ bounds bind) changes; the warm-start
carries that discrete state across solves.

In [2]:
class SimplexProjection:
    """min ½‖x − p‖² s.t. sum(x) = 1, x ≥ 0."""

    def __init__(self, n):
        self.n = n
        self.p = np.zeros(n)

    def set_parameter(self, p):
        self.p = np.asarray(p, dtype=float)

    def objective(self, x):
        return 0.5 * float((x - self.p) @ (x - self.p))

    def gradient(self, x):
        return x - self.p

    def constraints(self, x):
        return np.array([float(x.sum())])

    def jacobianstructure(self):
        return (np.zeros(self.n, dtype=np.int64), np.arange(self.n, dtype=np.int64))

    def jacobian(self, x):
        return np.ones(self.n)

    def hessianstructure(self):
        idx = np.arange(self.n, dtype=np.int64)
        return (idx, idx)

    def hessian(self, x, lagrange, obj_factor):
        return np.full(self.n, obj_factor)


def build_problem(nlp, algorithm):
    p = pounce.Problem(
        n=nlp.n, m=1, problem_obj=nlp,
        lb=[0.0] * nlp.n, ub=[1e20] * nlp.n,
        cl=[1.0], cu=[1.0],
    )
    p.add_option('algorithm', algorithm)
    p.add_option('print_level', 0)
    return p

## Step 1: cold IPM solve at $p_0$

Picking $p_0 = (0.5, 0.4, -0.1)$ — the negative coordinate forces
$x_3 \ge 0$ to be active at the optimum.

In [3]:
nlp = SimplexProjection(n=3)
nlp.set_parameter([0.5, 0.4, -0.1])

ipm_prob = build_problem(nlp, 'interior-point')
x_ipm, info_ipm = ipm_prob.solve(x0=np.full(3, 1.0 / 3))

print('status:    ', info_ipm['status_msg'])
print('x*:        ', x_ipm)
print('mult_g:    ', info_ipm['mult_g'])
print('mult_x_L:  ', info_ipm['mult_x_L'])
print('mult_x_U:  ', info_ipm['mult_x_U'])
print('g(x*):     ', info_ipm['g'])

status:     Solve_Succeeded
x*:         [5.49999940e-01 4.49999941e-01 1.19071471e-07]
mult_g:     [-0.04999994]
mult_x_L:   [4.40141312e-09 5.34423845e-09 5.00001835e-02]
mult_x_U:   [0. 0. 0.]
g(x*):      [1.]


## Step 2: classify the active set

`pounce.classify_working_set` takes the converged primal, the
multipliers (`mult_g`, `mult_x_L`, `mult_x_U`), the constraint
values $g(x^\star)$, and the bound vectors, and returns a
`(bounds, constraints)` tuple of int8 numpy arrays — the same
encoding `Problem.solve(..., working_set=ws)` consumes.

Status codes:

| code | meaning                                        |
|------|------------------------------------------------|
| 0    | Inactive (strictly interior)                  |
| 1    | AtLower (active at lower bound)               |
| 2    | AtUpper (active at upper bound)               |
| 3    | Fixed (variables) or Equality (constraints)   |

In [4]:
ws = pounce.classify_working_set(
    x=x_ipm,
    x_l=np.zeros(3), x_u=np.full(3, 1e20),
    g=info_ipm['g'],
    g_l=np.array([1.0]), g_u=np.array([1.0]),
    lambda_g=info_ipm['mult_g'],
    z_l=info_ipm['mult_x_L'],
    z_u=info_ipm['mult_x_U'],
    m_eq=1,
)
bounds, cons = ws
print('bounds:', bounds.tolist())
print('cons:  ', cons.tolist())

bounds: [0, 0, 1]
cons:   [3]


## Step 3: perturb $p$ and run the SQP corrector

Apply a small step $\Delta p = (0.02, -0.01, 0.05)$. The active set is
stable across this perturbation — $x_3$ still binds at zero — so
the SQP corrector should converge in one outer iteration.

In [5]:
nlp.set_parameter([0.52, 0.39, -0.05])

sqp_prob = build_problem(nlp, 'active-set-sqp')
x_sqp, info_sqp = sqp_prob.solve(x0=x_ipm, working_set=ws)

print('SQP corrector')
print('  status:    ', info_sqp['status_msg'])
print('  x*:        ', x_sqp)
print('  iter_count:', info_sqp['iter_count'])  # SQP outer iters
print('  ws:        ', info_sqp['working_set'])

# Check the closed-form predictor: x_pred_i = max(p_i - λ, 0) with
# λ such that sum equals 1.
p_new = nlp.p.copy()
lam = (p_new[0] + p_new[1] - 1.0) / 2.0   # x_3 active ⇒ 2 interior vars
x_predictor = np.array([p_new[0] - lam, p_new[1] - lam, 0.0])
print('  closed form:', x_predictor)
print('  |err|_inf:  {:.2e}'.format(np.max(np.abs(x_sqp - x_predictor))))

SQP corrector
  status:     Solve_Succeeded
  x*:         [0.565 0.435 0.   ]
  iter_count: 1
  ws:         (array([0, 0, 1], dtype=int8), array([3], dtype=int8))
  closed form: [0.565 0.435 0.   ]
  |err|_inf:  1.11e-16


## Step 4: full parametric sweep — cold vs warm comparison

Sweep $p$ around the simplex centre on a circle of radius $r=0.2$
for 20 steps. At each step we run three solves and record the
iteration count:

1. **Cold IPM** — fresh start.
2. **Cold SQP** — fresh start, no working-set warm.
3. **Warm SQP** — working set carried from previous step.

For this convex QP the per-solve SQP cost is 1 outer iteration in
all modes (the QP _is_ the original problem); the warm-start
advantage shows in the *QP-internal* active-set add/drop count,
which the iter-count field doesn't surface. The notebook is
therefore an **API-correctness demo** — for a benchmark-grade
comparison point at a nonlinear MPC sequence and watch the
iteration-count delta.

In [6]:
n = 8
n_steps = 20
centre = np.full(n, 1.0 / n)
rng = np.random.default_rng(0)
direction = rng.standard_normal(n)
direction -= direction.mean()
direction /= np.linalg.norm(direction)
radius = 0.2

cold_ipm, cold_sqp, warm_sqp = [], [], []
last_ws = None

for k in range(n_steps):
    theta = 2.0 * np.pi * k / n_steps
    p_k = centre + radius * np.cos(theta) * direction
    nlp_k = SimplexProjection(n=n)
    nlp_k.set_parameter(p_k)

    _, info_a = build_problem(nlp_k, 'interior-point').solve(x0=centre.copy())
    cold_ipm.append(info_a['iter_count'])

    _, info_b = build_problem(nlp_k, 'active-set-sqp').solve(x0=centre.copy())
    cold_sqp.append(info_b['iter_count'])

    kwargs = {'working_set': last_ws} if last_ws is not None else {}
    _, info_c = build_problem(nlp_k, 'active-set-sqp').solve(x0=centre.copy(), **kwargs)
    warm_sqp.append(info_c['iter_count'])
    last_ws = info_c['working_set']

print(f'mean iter count over {n_steps} steps:')
print(f'  cold IPM: {np.mean(cold_ipm):.2f}')
print(f'  cold SQP: {np.mean(cold_sqp):.2f}')
print(f'  warm SQP: {np.mean(warm_sqp):.2f}')

mean iter count over 20 steps:
  cold IPM: 6.65
  cold SQP: 0.90
  warm SQP: 0.90


## Where to next

- **C ABI version**: `crates/pounce-cinterface/examples/sqp_warm_start.c`
  is the same pattern in C against `IpoptSetWarmStartWorkingSet` /
  `IpoptGetWorkingSet` / `IpoptSolveWarmStart`.
- **GAMS version**: `gams/examples/parametric_sqp_warm_start.gms`
  demonstrates the marginal-based carry across `Solve` statements,
  plus the §7.4(b) persistent state-file option.
- **Sensitivity hand-off**: `SensSolve` returns the converged user-
  space multipliers on `SensResult` so the parametric predictor +
  SQP corrector pattern is a single sens solve plus one call to
  `pounce.classify_working_set`. See `crates/pounce-sensitivity/src/convenience.rs`
  module docstring for the full pipeline.
- **Options reference**: `docs/tutorials/active-set-sqp.md` §2 lists
  every `sqp_*` option.
- **Design rationale**: `docs/research/active-set-sqp-warm-start.md`
  for the algorithmic choices and references.